# GPT-5 — Data Collection (Repeatability)

This notebook collects the **repeatability** of the GPT-5 response corpus used in the paper. 

**Protocol (identical to the Gemini 3 Flash collection notebook):**
- Model: `gpt-5` (OpenAI; released August 7, 2025)
- Temperature: 1.0, top_p: 1.0
- Runs per prompt: 30
- Max concurrent requests: 5 (asyncio semaphore)
- Per-prompt inter-run spacing: 60 seconds within the same prompt
- Up to 6 retry attempts per API call
- Continuation logic if the response ends with `finish_reason == 'length'` (up to 2 extra calls with "Continue.")
- Parallel across the 15 prompts; sequential across the 30 runs within a prompt


> **Reviewer note:** To reproduce this notebook, update `PROJECT_DIR` to your local path. API keys are read from a `.env` file in `PROJECT_DIR` (see the `.env.example` in the repository root).

In [ ]:
# ---------- Load API key from .env (GitHub release version) ----------
# The hardcoded API key has been replaced with an environment-variable read.
# Place a .env file in PROJECT_DIR containing:  OPENAI_API_KEY=sk-...
import os
from pathlib import Path

PROJECT_DIR = Path('/Users/luizzadallapria/Documents/GPT5_Study')

def load_env(env_path):
    with open(env_path) as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('#'):
                continue
            if '=' in line:
                k, v = line.split('=', 1)
                k = k.strip()
                if k.lower() == 'openai_api_key':
                    k = 'OPENAI_API_KEY'
                os.environ[k] = v.strip().strip('"').strip("'")

load_env(PROJECT_DIR / '.env')
assert os.environ.get('OPENAI_API_KEY'), 'OPENAI_API_KEY missing from .env'
print('API key loaded ✓')


In [ ]:
from openai import OpenAI
import pandas as pd
import time
import os
import csv
import datetime

# ---------------
# Amelia Changes
# ---------------
import threading
import concurrent.futures
import asyncio
# this next line can be added with the earlier one once we decide on the version using
from openai import AsyncOpenAI
# ---------------

In [ ]:
df_prompts = pd.read_csv("repeatability_prompts.csv")
prompts = list(df_prompts.itertuples(index=False, name=None))

In [ ]:
# CONFIG
MODEL_NAME   = "gpt-5"     # choose the exact model you’ll use
TEMPERATURE  = 1
NUM_RUNS     = 30                  # per prompt
CSV_PATH     = "official_repeatability_results.csv"

# -----------------
# amelia's changes
# -----------------
# max concurrent requests that can hit the Chat API at once
MAX_CONCURRENT_REQUESTS = 5

# Initialize client
# ---------------
# Amelia's changes
# -----------
client = AsyncOpenAI(api_key=os.environ["OPENAI_API_KEY"])

# Ensure CSV has a header
header = [
    "prompt_id","prompt_text","run_id",
    "model","temperature",
    "timestamp_utc","latency_ms",
    "prompt_tokens","completion_tokens","total_tokens",
    "finish_reason","answer_content"
]

if not os.path.exists(CSV_PATH):
    with open(CSV_PATH, "w", newline="", encoding="utf-8") as f:
        csv.writer(f).writerow(header)


def _extract_text(choice):
    """
    Robustly get text from a chat.completions choice.
    Never stringify the message object.
    """
    txt = getattr(choice.message, "content", None)
    if isinstance(txt, str) and txt.strip():
        return txt.strip()

    # Rare case: list of parts
    if isinstance(txt, list):
        parts = []
        for part in txt:
            if isinstance(part, dict):
                if "text" in part and isinstance(part["text"], str):
                    parts.append(part["text"])
                elif "content" in part and isinstance(part["content"], str):
                    parts.append(part["content"])
        if parts:
            return "\n".join(parts).strip()

    # No text available
    return ""


# -----------------
# Amelia's changes
# -----------------
async def call_model(prompt_text, sem: asyncio.Semaphore):
    """
    Async version of call_model:
    - continuation if finish_reason == 'length'
    - per-request concurrency control via semaphore
    """
    t0 = time.time()

    messages = [{"role": "user", "content": prompt_text}]
    collected = []

    total_prompt_tokens = 0
    total_completion_tokens = 0
    total_tokens = 0

    finish_reason = None
    MAX_CONTINUES = 2
    REQUEST_TIMEOUT = 60

    # same model call pattern you used
    # for loop to cycle through the prompts
    # Main loop: for each prompt, run NUM_RUNS with 5s spacing
    for _ in range(MAX_CONTINUES + 1):
        async with sem:
            resp = await client.chat.completions.create(
                model=MODEL_NAME,
                temperature=TEMPERATURE,
                top_p=1.0,
                messages=messages,
                timeout=REQUEST_TIMEOUT,
            )

        choice = resp.choices[0]
        usage  = getattr(resp, "usage", None)

        text = _extract_text(choice)
        collected.append(text)

        # TODO CHECK THIS
        if usage:
            total_prompt_tokens     += getattr(usage, "prompt_tokens", 0)
            total_completion_tokens += getattr(usage, "completion_tokens", 0)
            total_tokens            += getattr(usage, "total_tokens", 0)

        # TODO CHECK THIS
        finish_reason = getattr(choice, "finish_reason", None)
        if finish_reason != "length":
            break

        # TODO CHECK THIS
        # If we hit length, ask it to continue
        messages.append({"role": "assistant", "content": text})
        messages.append({"role": "user", "content": "Continue."})

    final_text = " ".join(collected).strip()

    # Fallback via Responses API if got nothing
    if not final_text:
        try:
            async with sem:
                r = await client.responses.create(
                    model=MODEL_NAME,
                    input=[{"role": "user", "content": prompt_text}],
                    timeout=REQUEST_TIMEOUT,
                )
            # Prefer output_text; fall back to any available field just in case
            final_text = (getattr(r, "output_text", "") or "").strip()
        except Exception:
            # leave as empty if fallback also fails
            pass

    t1 = time.time()

    return {
        "answer": final_text,
        "finish_reason": finish_reason,
        "latency_ms": int((t1 - t0) * 1000),
        "prompt_tokens": total_prompt_tokens,
        "completion_tokens": total_completion_tokens,
        "total_tokens": total_tokens,
    }


async def run_prompt(
    prompt_id: int,
    prompt_text: str,
    sem: asyncio.Semaphore,
    csv_lock: asyncio.Lock,
):
    """
    Runs NUM_RUNS for a single (prompt_id, prompt_text),
    ensuring at least 5 seconds between runs of THIS prompt.
    """
    print(f"\n=== Prompt {prompt_id}: collecting {NUM_RUNS} runs ===")

    for run_id in range(1, NUM_RUNS + 1):
        try:
            out = await call_model(prompt_text, sem)

            row = [
                prompt_id, prompt_text, run_id,
                MODEL_NAME, TEMPERATURE,
                datetime.datetime.utcnow().isoformat(),
                out["latency_ms"],
                out["prompt_tokens"], out["completion_tokens"], out["total_tokens"],
                out["finish_reason"], out["answer"],
            ]

            # single-writer section for CSV
            async with csv_lock:
                with open(CSV_PATH, "a", newline="", encoding="utf-8") as f:
                    csv.writer(f).writerow(row)

            print(
                f"Successful prompt {prompt_id} Run {run_id:02d} saved "
                f"(latency {out['latency_ms']} ms, tokens {out['total_tokens']})"
            )

        except Exception as e:
            print(f"Failed Prompt and will repeat soon; {prompt_id} Run {run_id:02d} error: {e}")
            # light backoff on error!!!! Change this to 60 if keep running into errors
            await asyncio.sleep(5)
            continue

        # enforceing >= 5 seconds between repeats of THIS prompt
        await asyncio.sleep(5)


async def run_entire_code():
    """
    this is basically the main function

    we just broke all the parts that are called commonly down so its easier to read, feel free to revert this if you want

    this is also the mechanism that controls how many concurrent requests are, and when the locking and gathering takes place
    """
    sem = asyncio.Semaphore(MAX_CONCURRENT_REQUESTS)
    # making sure everything stays seperate
    csv_lock = asyncio.Lock()

    # creating tasks (aka running all the prompts)
    tasks = [
        asyncio.create_task(
            run_prompt(prompt_id, prompt_text, sem, csv_lock)
        )
        for prompt_id, prompt_text in prompts
    ]

    # should now wait until all tasks are completed
    await asyncio.gather(*tasks)
    print(f"\n✅ Done. Results saved to {CSV_PATH}") # quick note, the emoji is so cute!


await run_entire_code()

# ------------------------



=== Prompt 1: collecting 30 runs ===

=== Prompt 2: collecting 30 runs ===

=== Prompt 3: collecting 30 runs ===

=== Prompt 4: collecting 30 runs ===

=== Prompt 5: collecting 30 runs ===

=== Prompt 6: collecting 30 runs ===

=== Prompt 7: collecting 30 runs ===

=== Prompt 8: collecting 30 runs ===

=== Prompt 9: collecting 30 runs ===

=== Prompt 10: collecting 30 runs ===

=== Prompt 11: collecting 30 runs ===

=== Prompt 12: collecting 30 runs ===

=== Prompt 13: collecting 30 runs ===

=== Prompt 14: collecting 30 runs ===

=== Prompt 15: collecting 30 runs ===


/tmp/ipython-input-1423800010.py:163: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  datetime.datetime.utcnow().isoformat(),


Successful prompt 2 Run 01 saved (latency 34339 ms, tokens 1684)
Successful prompt 5 Run 01 saved (latency 44485 ms, tokens 1539)
Successful prompt 3 Run 01 saved (latency 46661 ms, tokens 1550)
Successful prompt 1 Run 01 saved (latency 48793 ms, tokens 1621)
Successful prompt 7 Run 01 saved (latency 82564 ms, tokens 1466)
Successful prompt 8 Run 01 saved (latency 84749 ms, tokens 1340)
Successful prompt 9 Run 01 saved (latency 94610 ms, tokens 1477)
Successful prompt 12 Run 01 saved (latency 109322 ms, tokens 729)
Successful prompt 11 Run 01 saved (latency 138089 ms, tokens 1729)
Successful prompt 10 Run 01 saved (latency 139589 ms, tokens 2040)
Successful prompt 13 Run 01 saved (latency 154326 ms, tokens 1366)
Successful prompt 4 Run 01 saved (latency 181435 ms, tokens 2049)
Successful prompt 14 Run 01 saved (latency 187646 ms, tokens 1671)
Successful prompt 2 Run 02 saved (latency 170568 ms, tokens 1891)
Failed Prompt and will repeat soon; 6 Run 01 error: Request timed out.
Successf